In [2]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [ ]:
import re
from groq import Groq

def IntentClassifier(user_message):
  client = Groq(api_key=api_key)

  system_message = {
              "role": "system",
              "content": '''
                User: What are some coping strategies for stress?
                Intent: asking_mental_health_question

                User: Thank you very much
                Intent: gratitude

                User: Who won the World Cup?
                Intent: out_of_scope

                User: Good morning
                Intent: greeting

                User: I have been feeling depressed recently
                Intent: asking_mental_health_question

                User: See you later
                Intent: goodbye

                User: Write a Python program to sort a list
                Intent: out_of_scope

                User: Hello
                Intent: greeting

                User: I appreciate your support
                Intent: gratitude

                User: What's the capital of France?
                Intent: out_of_scope

                User: Bye
                Intent: goodbye

                User: How can I manage anxiety?
                Intent: asking_mental_health_question
                '''}
  user_message = {
              "role": "user",
              "content": f'''
              Classify the intent of the following message.
              {user_message}
              Intent:
              '''}

  chat_completion = client.chat.completions.create( messages=[system_message, user_message], model = "openai/gpt-oss-20b")
  match = re.search(
    r"(greeting|goodbye|gratitude|asking_mental_health_question|out_of_scope)", chat_completion.choices[0].message.content)

  if match:
    res = match.group(1)
  else:
    res = "None"

  intent_to_id = {
    "greeting": 0,
    "goodbye": 1,
    "gratitude": 2,
    "asking_mental_health_question": 3,
    "out_of_scope": 4}

  if res not in intent_to_id.keys():
    response = {}
    response["role"] = chat_completion.choices[0].message.role
    response["content"] = chat_completion.choices[0].message.content

    second_user_message = {"role":"user",
                           "content": "you have to output just the class belongs into greeting, goodbye, gratitude,asking_mental_health_question, out_of_scope. and the output should be  intent: class"}

    chat_completion = client.chat.completions.create( messages=[system_message, user_message, response, second_user_message], model = "openai/gpt-oss-20b")
    match = re.search(r"(greeting|goodbye|gratitude|asking_mental_health_question|out_of_scope)", chat_completion.choices[0].message.content)

    if match:
      res = match.group(1)
    else:
      res = "None"

    if res not in intent_to_id.keys():
      return -1
    else:
      return intent_to_id[res]
  else:
    return intent_to_id[res]

In [ ]:
intent_to_id = {
    "greeting": 0,
    "goodbye": 1,
    "gratitude": 2,
    "asking_mental_health_question": 3,
    "out_of_scope": 4,
}

def IntentClassifier2(user_text):
    client = Groq(api_key=api_key)

    system_message = {
        "role": "system",
        "content": """
You are an intent classifier.

Possible intents:
- greeting
- goodbye
- gratitude
- asking_mental_health_question
- out_of_scope

Example:

User: How can I manage anxiety?
Intent: asking_mental_health_question

Rules:
1. Output exactly one intent.
2. Do not explain.
3. Do not output any extra text.
"""
    }

    user_message = {
        "role": "user",
        "content": f"""
User: {user_text}
Intent:
"""
    }

    chat_completion = client.chat.completions.create(
        messages=[system_message, user_message],
        model="openai/gpt-oss-20b",
        temperature=0
    )

    response = chat_completion.choices[0].message.content.strip()

    match = re.search(
        r"(greeting|goodbye|gratitude|asking_mental_health_question|out_of_scope)",
        response
    )

    if match:
        return intent_to_id[match.group(1)]

    return -1

In [5]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_json("hf://datasets/Amod/mental_health_counseling_conversations/combined_dataset.json", lines=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3512 entries, 0 to 3511
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Context   3512 non-null   object
 1   Response  3512 non-null   object
dtypes: object(2)
memory usage: 55.0+ KB


In [6]:
import time

c_1 = 0
c_2 = 0
for i,text in enumerate(df['Context'].sample(n=100)):
  print(i+1)
  if IntentClassifier(text) != 3:
    c_1 +=1
  if IntentClassifier2(text) != 3:
    c_2 +=1
  time.sleep(0.1)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100


In [7]:
print(c_1,c_2)

4 14
